In [1]:
from transformers import pipeline,AutoModelForSeq2SeqLM,Trainer,TrainingArguments,AutoTokenizer
from datasets import load_dataset
from huggingface_hub import notebook_login
import numpy as np
import pandas as pd
import torch
from collections import defaultdict

In [2]:
from datasets import load_dataset
dataset = load_dataset("Helsinki-NLP/kde4",lang1="en",lang2="fr")

In [3]:
split_datasets = dataset["train"].train_test_split(train_size=0.9, seed=20)

In [4]:
split_datasets['train'][0]

{'id': '92924',
 'translation': {'en': "Calibration is about to check the value range your device delivers. Please move axis %1 %2 on your device to the maximum position. Press any button on the device or click on the'Next 'button to continue with the next step.",
  'fr': "Le calibrage va vérifier la plage de valeurs que votre matériel produit. Veuillez déplacer l'axe %1 %2 de votre périphérique à la position maximale. Appuyez sur n'importe quel bouton du périphérique ou sur le bouton « & #160; Suivant & #160; » pour la prochaine étape."}}

In [5]:
model_checkpoint="Helsinki-NLP/opus-mt-en-fr"
tokenizer=AutoTokenizer.from_pretrained(model_checkpoint)
model=AutoModelForSeq2SeqLM.from_pretrained(model_checkpoint)
inputs=tokenizer("Peace be upon you", return_tensors="pt")

/usr/local/lib/python3.13/dist-packages/transformers/models/marian/tokenization_marian.py:176: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")


Loading weights:   0%|          | 0/258 [00:00<?, ?it/s]

In [6]:
inputs

{'input_ids': tensor([[5831,   45, 1185,   55,    0]]), 'attention_mask': tensor([[1, 1, 1, 1, 1]])}

In [7]:
output=model.generate(**inputs)

In [8]:
tokenizer.decode(output[0],skip_special_tokens=True)

'Que la paix soit sur toi'

In [9]:
max_length=128
def preprocess_text(dataset):
    inputs=[ex['en'] for ex in dataset['translation']]
    targets=[ex['fr'] for ex in dataset['translation']]
    model_inputs=tokenizer(inputs,text_target=targets,max_length=max_length,truncation=True)
    return model_inputs

In [10]:
tokenized_dataset=split_datasets.map(preprocess_text,batched=True,remove_columns=split_datasets['train'].column_names)

In [11]:
tokenized_dataset

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 189155
    })
    test: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 21018
    })
})

In [12]:
from transformers import DataCollatorForSeq2Seq
data_collator=DataCollatorForSeq2Seq(tokenizer=tokenizer,model=model)

In [13]:
%pip install --upgrade sacrebleu

In [14]:
import evaluate
metric = evaluate.load("sacrebleu")

In [15]:
def compute_metrics(eval_pred):
    preds,labels=eval_pred

    decoded_preds=tokenizer.batch_decode(preds,skip_special_tokens=True)
    labels=np.where(labels!=-100,labels,tokenizer.pad_token_id)
    decoded_labels=tokenizer.batch_decode(labels,skip_special_tokens=True)
    clean_preds=[pred.strip() for pred in decoded_preds]
    clean_labels=[[label.strip()] for label in decoded_labels]       ###A particular input can have multiple correct translations###

    result=metric.compute(predictions=decoded_preds,references=decoded_labels)
    return {"bleu":result['score']}


In [16]:
from transformers import Seq2SeqTrainingArguments,Seq2SeqTrainer

In [17]:
args=Seq2SeqTrainingArguments("marian-finetuned-kde4-en-to-fr",eval_strategy="no",save_strategy="epoch",num_train_epochs=3,
                              learning_rate=2e-5,weight_decay=0.01,per_device_train_batch_size=64,per_device_eval_batch_size=32,predict_with_generate=True,
                              fp16=True,push_to_hub=True)

In [18]:
##train_size=20_000
##test_size=5_000
##split_test_dataset=tokenized_dataset['train'].train_test_split(train_size=train_size,test_size=test_size)

In [19]:
trainer=Seq2SeqTrainer(model,args,train_dataset=tokenized_dataset['train'],eval_dataset=tokenized_dataset['test'],data_collator=data_collator,
                       processing_class=tokenizer,compute_metrics=compute_metrics)

In [20]:
trainer.evaluate(max_length=max_length)

Training Loss,Validation Loss,Step,Bleu
No log,1.705812,0,39.259094


{'eval_loss': 1.7058119773864746, 'eval_bleu': 39.2590936022617}

In [21]:
trainer.train()

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.


Step,Training Loss
500,1.337207
1000,1.198533
1500,1.124122
2000,1.065040
2500,1.054469
3000,1.011665
3500,0.947316
4000,0.942528
4500,0.931363
5000,0.921713


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=8868, training_loss=0.9766968592470382, metrics={'train_runtime': 3128.0919, 'train_samples_per_second': 181.409, 'train_steps_per_second': 2.835, 'total_flos': 1.4078773000470528e+16, 'train_loss': 0.9766968592470382, 'epoch': 3.0})

In [22]:
trainer.evaluate(max_length=max_length)

Training Loss,Validation Loss,Step,Bleu
0.859783,0.880722,8868,52.161832


{'eval_loss': 0.8807216286659241, 'eval_bleu': 52.16183164524019}

In [23]:
trainer.push_to_hub()

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

CommitInfo(commit_url='https://huggingface.co/Adnan2942/marian-finetuned-kde4-en-to-fr/commit/5a585bd9a9db9f958efcc23ee5ea0d1f83d1f76f', commit_message='End of training', commit_description='', oid='5a585bd9a9db9f958efcc23ee5ea0d1f83d1f76f', pr_url=None, repo_url=RepoUrl('https://huggingface.co/Adnan2942/marian-finetuned-kde4-en-to-fr', endpoint='https://huggingface.co', repo_type='model', repo_id='Adnan2942/marian-finetuned-kde4-en-to-fr'), pr_revision=None, pr_num=None)